In [1]:
from __future__ import annotations

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from pydantic import BaseModel, Field, conint, field_validator, ConfigDict
import sys
from typing import Any, Dict, List, Tuple, Optional, Literal
from dataclasses import dataclass
import os, io, re, json, time, hashlib, datetime as dt
import fitz

load_dotenv(override=True)
from pprint import pprint


### PDF Ingest Tool

In [2]:
PARSER_VERSION = "ingest_v1.0"

def sha256_of_file(path: str, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def ts_now() -> str:
    return dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat() + "Z"

def rect_overlap_horiz(a: fitz.Rect, b: fitz.Rect) -> float:
    inter = max(0.0, min(a.x1, b.x1) - max(a.x0, b.x0))
    if a.width == 0 or b.width == 0:
        return 0.0
    return inter / max(1e-9, min(a.width, b.width))

_CAP_RE = re.compile(r"^\s*(fig(?:ure)?\.?\s*\d+[:\.\s])", re.IGNORECASE)
_FREQ_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(kHz|MHz|GHz)", re.IGNORECASE)
_UNIT_FACTORS = {"khz": 1e3, "mhz": 1e6, "ghz": 1e9}

def normalize_frequencies(text: str) -> List[float]:
    freqs: List[float] = []
    for m in _FREQ_RE.finditer(text):
        val = float(m.group(1))
        unit = m.group(2).lower()
        freqs.append(val * _UNIT_FACTORS[unit])
    # de-dup preserve order
    seen, out = set(), []
    for hz in freqs:
        if hz not in seen:
            seen.add(hz)
            out.append(hz)
    return out

def detect_multicol(blocks: List[Tuple[float, float, float, float, str]]) -> bool:
    if not blocks:
        return False
    page_min = min(b[0] for b in blocks)
    page_max = max(b[2] for b in blocks)
    mid = (page_min + page_max) / 2.0
    centers = [(b[0] + b[2]) / 2.0 for b in blocks]
    left  = [x for x in centers if x <  mid]
    right = [x for x in centers if x >= mid]
    return len(left) >= 3 and len(right) >= 3

@dataclass
class FigureEntry:
    figure_id: str
    page: int
    bbox: Tuple[float, float, float, float]
    caption: str | None
    caption_preview: str | None
    image_path: str
    warnings: List[str]

def _extract_page_text(page: fitz.Page) -> tuple[str, list[tuple[float,float,float,float,str]]]:
    blocks = page.get_text("blocks")
    blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))
    text = "\n".join(b[4].strip() for b in blocks_sorted if b[4].strip())
    text_blocks = [(b[0], b[1], b[2], b[3], b[4]) for b in blocks_sorted if b[4].strip()]
    return text, text_blocks

def _find_caption_near_image(page: fitz.Page,
                             text_blocks: list[tuple[float,float,float,float,str]],
                             img_rect: fitz.Rect) -> tuple[str|None, str|None, list[str]]:
    warnings: List[str] = []
    # collect line-level boxes/text for better caption proximity
    lines_info = page.get_text("dict")
    line_spans: List[tuple[fitz.Rect, str]] = []
    for b in lines_info.get("blocks", []):
        if b.get("type", 0) != 0:
            continue
        for l in b.get("lines", []):
            xs0 = [s["bbox"][0] for s in l.get("spans", []) if "bbox" in s]
            ys0 = [s["bbox"][1] for s in l.get("spans", []) if "bbox" in s]
            xs1 = [s["bbox"][2] for s in l.get("spans", []) if "bbox" in s]
            ys1 = [s["bbox"][3] for s in l.get("spans", []) if "bbox" in s]
            if not xs0 or not ys0 or not xs1 or not ys1:
                continue
            rect = fitz.Rect(min(xs0), min(ys0), max(xs1), max(ys1))
            text = "".join(s.get("text","") for s in l.get("spans", [])).strip()
            if text:
                line_spans.append((rect, text))

    below, above = [], []
    for rect, line in line_spans:
        if rect.y0 >= img_rect.y1 and rect.y0 <= img_rect.y1 + 80:
            if rect_overlap_horiz(rect, img_rect) >= 0.5:
                below.append((abs(rect.y0 - img_rect.y1), line))
        elif rect.y1 <= img_rect.y0 and rect.y1 >= img_rect.y0 - 60:
            if rect_overlap_horiz(rect, img_rect) >= 0.5:
                above.append((abs(img_rect.y0 - rect.y1), line))

    def pick(neigh: list[tuple[float,str]]) -> list[str]:
        neigh_sorted = sorted(neigh, key=lambda t: t[0])
        return [t[1] for t in neigh_sorted[:3]]

    candidates = pick(below) or pick(above)
    if not candidates:
        warnings.append("no_nearby_caption")
        return None, None, warnings

    caps = [c for c in candidates if _CAP_RE.match(c)]
    chosen = caps or candidates
    full = " ".join(chosen).strip()
    prev = full[:140] + ("â€¦" if len(full) > 140 else "")
    if not caps:
        warnings.append("caption_not_prefixed_by_figure")
    return full, prev, warnings

def _extract_figures(page: fitz.Page,
                     page_num: int,
                     figures_dir: str,
                     text_blocks: list[tuple[float,float,float,float,str]],
                     fig_start_index: int) -> tuple[list[FigureEntry], int]:
    entries: List[FigureEntry] = []
    pinfo = page.get_text("dict")
    image_rects: List[fitz.Rect] = [
        fitz.Rect(*b["bbox"]) for b in pinfo.get("blocks", []) if b.get("type",0) == 1 and "bbox" in b
    ]
    for rect in image_rects:
        fig_id = f"fig_{fig_start_index:03d}"
        fig_start_index += 1

        zoom = 300.0 / 72.0
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
        img_path = os.path.join(figures_dir, f"{fig_id}.png")
        pix.save(img_path)

        cap_full, cap_prev, warns = _find_caption_near_image(page, text_blocks, rect)
        entries.append(FigureEntry(
            figure_id=fig_id,
            page=page_num,
            bbox=(float(rect.x0), float(rect.y0), float(rect.x1), float(rect.y1)),
            caption=cap_full,
            caption_preview=cap_prev,
            image_path=img_path,
            warnings=warns,
        ))
    return entries, fig_start_index

# ---------------- core impl (always include 'cache') ----------------

def _empty_manifest(file_hash: str, pdf_abs: str, bundle_dir: str, FiguresPath: str) -> Dict[str, Any]:
    return {
        "bundle_id": file_hash,
        "parser_version": PARSER_VERSION,
        "status": "failed",
        "cache": "miss",
        "created_at": ts_now(),
        "pdf_path": pdf_abs,
        "page_count": 0,
        "figure_count": 0,
        "pages_multi_column": [],
        "used_ocr": False,
        "frequency_hz_candidates": [],
        "paths": {
            "bundle_dir": os.path.abspath(bundle_dir),
            "pages_dir": os.path.join(os.path.abspath(bundle_dir), "pages"),
            "figures_dir": os.path.join(os.path.abspath(bundle_dir), "figures"),
            "manifest_path": os.path.join(os.path.abspath(bundle_dir), "manifest.json"),
            "figures_index": os.path.join(os.path.abspath(bundle_dir), "figures", "index.json"),
        },
        "warnings": [],
    }

def _ensure_ingested_impl(pdf_path: str, store_dir: str = "ingest_store") -> Dict[str, Any]:
    # Validate inputs EARLY â€” and always include 'cache'
    if not isinstance(pdf_path, str) or not pdf_path.lower().endswith(".pdf"):
        return {
            "status": "failed", "cache": "miss",
            "error": "pdf_path must be a .pdf file"
        }
    if not os.path.isfile(pdf_path):
        return {
            "status": "failed", "cache": "miss",
            "error": f"file not found: {pdf_path}"
        }

    pdf_abs = os.path.abspath(pdf_path)
    file_hash = sha256_of_file(pdf_abs)
    bundle_dir = os.path.join(store_dir, file_hash)
    ensure_dir(store_dir)
    manifest_path = os.path.join(bundle_dir, "manifest.json")

    # Cache hit path â€” and force 'cache'='hit' even if the stored manifest didn't have it
    if os.path.isfile(manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as f:
            old = json.load(f)
        old["cache"] = "hit"
        # ensure status key exists for downstream prints
        old.setdefault("status", "ready")
        return old

    # Build (cache miss)
    ensure_dir(bundle_dir)
    pages_dir   = os.path.join(bundle_dir, "pages")
    figures_dir = os.path.join(bundle_dir, "figures")
    ensure_dir(pages_dir); ensure_dir(figures_dir)

    manifest = _empty_manifest(file_hash, pdf_abs, bundle_dir, figures_dir)
    manifest["status"] = "ready"   # optimistic; flip to failed on exception
    manifest["cache"]  = "miss"

    pages_multi_col: List[int] = []
    freq_candidates: List[float] = []
    figure_entries: List[FigureEntry] = []

    try:
        doc = fitz.open(pdf_abs)
        manifest["page_count"] = doc.page_count

        fig_index = 1
        for i in range(doc.page_count):
            page_no = i + 1
            page = doc.load_page(i)
            text, text_blocks = _extract_page_text(page)

            # Save page text
            with open(os.path.join(pages_dir, f"page_{page_no:04d}.txt"), "w", encoding="utf-8") as f:
                f.write(text)

            if detect_multicol(text_blocks):
                pages_multi_col.append(page_no)
            if text:
                freq_candidates.extend(normalize_frequencies(text))

            figs, fig_index = _extract_figures(page, page_no, figures_dir, text_blocks, fig_index)
            figure_entries.extend(figs)

        # Caption-based frequency hints
        for fe in figure_entries:
            if fe.caption:
                freq_candidates.extend(normalize_frequencies(fe.caption))

        # Dedup freq hints
        seen, unique_freqs = set(), []
        for hz in freq_candidates:
            if hz not in seen:
                seen.add(hz)
                unique_freqs.append(hz)

        # Write figures index
        figures_index_path = os.path.join(figures_dir, "index.json")
        with open(figures_index_path, "w", encoding="utf-8") as fidx:
            json.dump(
                [
                    {
                        "figure_id": fe.figure_id,
                        "page": fe.page,
                        "bbox": fe.bbox,
                        "caption": fe.caption,
                        "caption_preview": fe.caption_preview,
                        "image_path": os.path.relpath(fe.image_path, bundle_dir),
                        "warnings": fe.warnings,
                    }
                    for fe in figure_entries
                ],
                fidx, ensure_ascii=False, indent=2
            )

        manifest.update({
            "figure_count": len(figure_entries),
            "pages_multi_column": pages_multi_col,
            "frequency_hz_candidates": unique_freqs,
            "paths": {
                "bundle_dir": os.path.abspath(bundle_dir),
                "pages_dir": os.path.abspath(pages_dir),
                "figures_dir": os.path.abspath(figures_dir),
                "manifest_path": os.path.abspath(manifest_path),
                "figures_index": os.path.abspath(figures_index_path),
            },
        })

        with open(manifest_path, "w", encoding="utf-8") as mf:
            json.dump(manifest, mf, ensure_ascii=False, indent=2)

        return manifest

    except Exception as e:
        manifest["status"] = "failed"
        manifest["warnings"].append(f"ingest_error:{e!r}")
        # Try to persist manifest even on failure
        try:
            with open(manifest_path, "w", encoding="utf-8") as mf:
                json.dump(manifest, mf, ensure_ascii=False, indent=2)
        except Exception:
            pass
        return manifest
# ----------------------------- TOOL: ensure_ingested --------------------

@function_tool
def ensure_ingested(pdf_path: str,
                    store_dir: str = "ingest_store") -> Dict[str, Any]:
    """
    Parse a PDF into a cached bundle with page text and figure crops/captions. 
    Idempotent: returns existing bundle if already processed.
    Inputs:
      - pdf_path: path to a local PDF file (must exist).
      - store_dir: root folder for cached bundles (default 'ingest_store').

    Returns (manifest dict):
      {
        'bundle_id': <sha256>,
        'parser_version': 'ingest_v1.0',
        'status': 'ready'|'failed',
        'cache': 'hit'|'miss',
        'created_at': ISO8601,
        'pdf_path': <abs path>,
        'page_count': int,
        'figure_count': int,
        'pages_multi_column': [ints],
        'used_ocr': false,
        'frequency_hz_candidates': [floats],
        'paths': {
          'bundle_dir': <str>,
          'pages_dir': <str>,
          'figures_dir': <str>,
          'manifest_path': <str>,
          'figures_index': <str>
        },
        'warnings': [str]
      }
    """
    return _ensure_ingested_impl(pdf_path=pdf_path, store_dir=store_dir)

    
        

## Structured Outputs

In [3]:
class StrictBaseModel(BaseModel):
    model_config = ConfigDict(extra='forbid')  # keep objects closed

class Point3D(StrictBaseModel):
    x: Optional[float] = None
    y: Optional[float] = None
    z: Optional[float] = None


class Provenance(StrictBaseModel):
    source_file: str = Field(description="e.g., 'pages/page_0006.txt' or 'figures/fig_003.png'")
    page_or_fig: str = Field(description="e.g., 'page 6', 'fig 3'")
    snippet: Optional[str] = Field(default=None, description="Short quote used as evidence")
    confidence: Optional[float] = Field(default=None, ge=0, le=1)


class MaterialProps(StrictBaseModel):
    name: Optional[str] = Field(default=None, description="e.g., 'Rogers RO3010'")
    epsilon_r: Optional[float] = Field(default=None, description="Relative permittivity εr")
    tan_delta: Optional[float] = Field(default=None, description="Loss tangent")
    conductivity_S_per_m: Optional[float] = Field(default=None, description="Metal conductivity if applicable")


class Layer(StrictBaseModel):
    order: int = Field(description="0 = bottom-most in the stackup (in +z order)")
    role: str = Field(description="Functional role of the layer")
    material: MaterialProps = Field(description="Material and its RF properties if available")
    z0: float = Field(description="Height from where the layer begins (z-axis) in mm. If this is the bottom layer, z0=0.")
    thickness_mm: Optional[float] = Field(default=None, description="Physical thickness in mm")
    pattern: Optional[str] = Field(default=None, description="e.g., 'solid', 'etched', 'slotted'")
    geometry: str = Field(
        default=None,
        description="Describe the geometry of this layer. All relative to the one before. Describe as accurately as possible with the dimensions and shapes."
    )
    provenance: Optional[Provenance] = None


class Polarization(StrictBaseModel):
    type: str = Field(description="The type of polarization: e.g., 'linear', 'circular', 'elliptical', 'dual-linear', 'dual-circular', 'dual-elliptical'")
    sense: Optional[Literal["RHCP", "LHCP"]] = Field(
        default=None, description="When circular, the handedness"
    )


class OperatingPoint(StrictBaseModel):
    freq_GHz: float = Field(description="Operating or reported frequency in GHz")
    gain_dBi: Optional[float] = Field(default=None, description="Reported gain in dBi at the operating frequency if applicable. If not stated, use None")
    axial_ratio_dB: Optional[float] = Field(default=None, description="Reported axial ratio in dB at the operating frequency if applicable. If not stated, use None")
    bandwidth_MHz: Optional[float] = Field(default=None, description="3-dB AR or |S11| bandwidth if stated or infered from images")
    provenance: Optional[Provenance] = None


class Feed(StrictBaseModel):
    type: str = Field(description="Type of feed used, e.g., 'coax_probe', 'microstrip_line', 'cpw', 'aperture_coupled', 'proximity_coupled'. If not stated, describe it. Don't forget to mention the materials used if needed and the dimensions of the feed or port used.")
    location_mm: Point3D = Field(default=None, description="Coordinates relative to patch/board origin, e.g., {'x': 2.5, 'y': 0, 'z': 0}, all in mm. Try to be as accurate as possible. If you can't provide, try to infer.")
    


class Antenna(StrictBaseModel):
    antenna_type: str = Field(description="The type of antenna e.g., 'microstrip patch', 'dipole', 'array', 'DRA', etc.")
    polarization: Polarization = Field(description="Polarization type (and sense if circular)")
    operating_points: List[OperatingPoint] = Field(
        description="Per-frequency metrics to avoid misalignment"
    )
    stackup: List[Layer] = Field(description="Layered description from bottom (order=0) to top")
    feed: Optional[Feed] = None
    notes: Optional[str] = None

    @field_validator("operating_points")
    @classmethod
    def _non_empty_ops(cls, v):
        if len(v) == 0:
            raise ValueError("Provide at least one OperatingPoint, even if freq only.")
        return v


In [4]:
print(ensure_ingested)

FunctionTool(name='ensure_ingested', description="Parse a PDF into a cached bundle with page text and figure crops/captions. \nIdempotent: returns existing bundle if already processed.\nInputs:\n  - pdf_path: path to a local PDF file (must exist).\n  - store_dir: root folder for cached bundles (default 'ingest_store').\n\nReturns (manifest dict):\n  {\n    'bundle_id': <sha256>,\n    'parser_version': 'ingest_v1.0',\n    'status': 'ready'|'failed',\n    'cache': 'hit'|'miss',\n    'created_at': ISO8601,\n    'pdf_path': <abs path>,\n    'page_count': int,\n    'figure_count': int,\n    'pages_multi_column': [ints],\n    'used_ocr': false,\n    'frequency_hz_candidates': [floats],\n    'paths': {\n      'bundle_dir': <str>,\n      'pages_dir': <str>,\n      'figures_dir': <str>,\n      'manifest_path': <str>,\n      'figures_index': <str>\n    },\n    'warnings': [str]\n  }", params_json_schema={'properties': {'pdf_path': {'title': 'Pdf Path', 'type': 'string'}, 'store_dir': {'default':

## Tool #2
### Read file

In [5]:
@function_tool
def read_file(file_path: str) -> str:
    """Read a file and return its contents.
    Inputs:
    - file_path: path to a local text file (must exist).
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"File not found: {file_path}")
    except PermissionError:
        raise PermissionError(f"Permission denied: {file_path}")
    except Exception as e:
        raise RuntimeError(f"An unexpected error occurred while reading {file_path}: {e}")

In [6]:
print(read_file)

FunctionTool(name='read_file', description='Read a file and return its contents.\nInputs:\n- file_path: path to a local text file (must exist).', params_json_schema={'properties': {'file_path': {'title': 'File Path', 'type': 'string'}}, 'required': ['file_path'], 'title': 'read_file_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000283AE767880>, strict_json_schema=True, is_enabled=True)


## Tool #3
### Convert image into readable string

In [7]:
@function_tool
def read_figure(fig_path: str) -> str:
    """
    Read a figure image and return its base64-encoded string.

    Args:
        fig_path (str): Path to the local figure (PNG/JPG).

    Returns:
        str: Base64-encoded image string.
    """
    import base64, os

    try:
        if not os.path.exists(fig_path):
            raise FileNotFoundError(f"Figure not found: {fig_path}")

        with open(fig_path, "rb") as f:
            b64_img = base64.b64encode(f.read()).decode("utf-8")

        return b64_img

    except Exception as e:
        return f"Error reading figure: {e}"

print(read_figure)


FunctionTool(name='read_figure', description='Read a figure image and return its base64-encoded string.', params_json_schema={'properties': {'fig_path': {'description': 'Path to the local figure (PNG/JPG).', 'title': 'Fig Path', 'type': 'string'}}, 'required': ['fig_path'], 'title': 'read_figure_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000283AE767EC0>, strict_json_schema=True, is_enabled=True)


## Paths

In [8]:
PDF_PATH= r"C:\\Users\\Lenovo\\SynologyDrive\\antenna_automation\\agentic_automation\\article\\test_article.pdf"   
STORE_DIR = r"C:\\Users\\Lenovo\\SynologyDrive\\antenna_automation\\agentic_automation\\ingest_store"
MANIFEST_PATH = r"C:\\Users\\Lenovo\\SynologyDrive\\antenna_automation\\agentic_automation\\ingest_store\\6a1c4667b0063ec10d1d39dceff33770e0e7babcd1c8f2ec6d701fdcaa959957\\manifest.json"
RANDOM_PAGE_PATH = r"C:\\Users\\Lenovo\\SynologyDrive\\antenna_automation\\agentic_automation\\ingest_store\\6a1c4667b0063ec10d1d39dceff33770e0e7babcd1c8f2ec6d701fdcaa959957\\pages\\page_0001.txt"
RANDOM_IMAGE_PATH = r"C:\\Users\\Lenovo\\SynologyDrive\\antenna_automation\\agentic_automation\\ingest_store\\6a1c4667b0063ec10d1d39dceff33770e0e7babcd1c8f2ec6d701fdcaa959957\\figures\\fig_001.png"

In [9]:
# print(read_file(RANDOM_PAGE_PATH))

In [10]:
# print(_ensure_ingested_impl(PDF_PATH, store_dir=STORE_DIR))

In [11]:
# print(read_figure(RANDOM_IMAGE_PATH))

In [12]:
# # test_ingest.py

# # 1) First run: cache miss, extraction happens

# print("\n--- First run (should be cache miss) ---")
# m1 = _ensure_ingested_impl(PDF_PATH, store_dir=STORE_DIR)
# pprint({
#     "status": m1.get("status"),
#     "cache": m1.get("cache"),
#     "bundle_id": m1.get("bundle_id"),
#     "page_count": m1.get("page_count"),
#     "figure_count": m1.get("figure_count"),
#     "warnings": m1.get("warnings"),
#     "freq_hints": m1.get("frequency_hz_candidates"),
# })

# print("\n--- Second run (should be cache hit) ---")
# m2 = _ensure_ingested_impl(PDF_PATH, store_dir=STORE_DIR)
# pprint({
#     "status": m2.get("status"),
#     "cache": m2.get("cache"),
#     "bundle_id": m2.get("bundle_id"),
#     "page_count": m2.get("page_count"),
#     "figure_count": m2.get("figure_count"),
# })


In [13]:
antenna_info_retrieval_instructions = """You are an expert antenna engineer. You are given a PDF article that contains information about antennas. 
Your task is to extract information about the antenna mentioned in the article. 
 """

antenna_info_retrieval_instructions = f"""
ROLE
You are the Antenna Info Extractor. From a single PDF, build a complete Antenna object (the project Pydantic schema) using the available tools. Be precise, minimize tool calls, and include provenance wherever possible. Do not guess; use null for unknowns.

TOOLS
- ensure_ingested(pdf_path={PDF_PATH}, store_dir={STORE_DIR}): build or reuse the ingest store for this PDF.
- read_file(path): read a text or json file inside the store (e.g., pages/page_0006.txt, index.json, manifest.json).
- read_figure(fig_path): read a figure image inside the store (e.g., figures/fig_003.png) for visual extraction.

EXPECTED STORE LAYOUT
- STORE_DIR/hash/pages/page_0001.txt : page_XXXX.txt
- STORE_DIR/hash/figures/fig_001.png : fig_XXX.png
- STORE_DIR/hash/figures/index.json : index of figures
- STORE_DIR/hash/manifest.json 

OUTPUT
Return exactly one valid Antenna JSON (no extra prose). The schema is provided via output_type, so follow it strictly:
- antenna_type: string
- polarization: fields type and optional sense
- operating_points: list of OperatingPoint; at least one item
- stackup: list of Layer; order starts at 0 (bottom) and increases
- feed: optional
- notes: optional
For numeric or structural facts, add a Provenance when possible (source_file, page_or_fig, optional snippet and confidence).

UNITS AND CONVENTIONS (STRICT)
- Frequencies in GHz (convert from MHz if needed).
- Dimensions in mm.
- Gain in dBi.
- Axial ratio in dB.
- If circular polarization, set sense to RHCP or LHCP when stated; otherwise leave null.
- If a value is not present, leave the corresponding field as null (do not write the string "none").

RANGES AND GUARDS
- operating_points must not be empty.

WORKFLOW (PLAN â†’ ACT)
1) Ingest
- Call ensure_ingested(pdf_path=PDF_PATH, store_dir=STORE_DIR).
2) Map files
- Try read_file(STORE_DIR + "/manifest.json") first. If present, use it to find pages and figure references.
- If manifest is missing, assume standard names and iterate pages/page_0001.txt upward until missing.
3) Targeted reading (minimize calls)
- Prioritize pages likely to contain structure and performance.
- Keywords for type/architecture: antenna, microstrip, patch, dipole, array, slot, configuration, geometry, proposed antenna.
- Keywords for stack/materials: substrate, relative permittivity, epsilon r, ur, tan delta, loss tangent, ground plane, superstrate, encapsulation.
- Keywords for feed: coax, probe, SMA, microstrip feed, CPW, aperture coupled, proximity coupled.
- Keywords for performance: frequency, MHz, GHz, gain, axial ratio, AR, bandwidth, efficiency, S11, |S11|, return loss.
4) Use figures when helpful or when you are trying to resolve ambiguities. You can use it to recheck numbers or extract geometry and stackup.
- When a page references a figure for geometry or performance, check the caption in the STORE_DIR/hash/figures/index.json read_figure for the referenced PNG. 
5) Populate the Antenna object
- antenna_type: exact term used (e.g., microstrip patch).
- polarization.type: linear, circular, elliptical, dual-linear, dual-circular, or dual-elliptical if explicitly stated. polarization.sense only if stated.
- operating_points: add an entry for each clearly stated frequency; fill gain_dBi, axial_ratio_dB, bandwidth_MHz when available; otherwise null. Attach provenance.
- stackup: create Layer items from bottom to top. Fill role, material props, thickness, copper thickness, pattern, geometry, and provenance when available.
- feed: set type and add location_mm and size_mm if reported. Attach provenance when possible.
- notes: capture important caveats (variants, on-body vs free-space, measurement vs simulation).
6) Sanity checks
- Ensure operating_points is not empty.
- Verify conversions: MHz to GHz and dimensions in mm.
- Check stackup.order starts at 0 and increments with no gaps.
- Resolve conflicts by choosing the clearest evidence and mention alternatives in notes.
7) Return
- Return only the final Antenna JSON object. No explanations, no comments, no extra keys.

PROVENANCE RULES
- For text-derived values: include a short snippet (<25 words), the exact page path (e.g., pages/page_0006.txt), and a logical label (e.g., page 6).
- For figure-derived values: include the figure path (e.g., figures/fig_003.png) and logical label (e.g., fig 3). Snippet can be null.
- Set confidence between 0 and 1 to reflect certainty.

AMBIGUITY AND VARIANTS
- If multiple designs/variants exist, select the one explicitly named proposed antenna or the main design. Mention others briefly in notes.
- Do not invent materials; if substrate properties are omitted, leave them null.

EFFICIENCY GUIDELINES
- Prefer a few targeted read_file calls over exhaustive reading.
- Read figure captions and results pages first.
- Only call read_figure when a figure is referenced or contains numeric labels needed.
"""


In [14]:
tools = [ensure_ingested, read_figure, read_file]
information_extracter_agent = Agent(
    name = "Antenna Info Extracter", 
    instructions = antenna_info_retrieval_instructions, 
    tools = tools, 
    model = "gpt-4o-mini",
    output_type= Antenna
    )


In [15]:
with trace("Extract antenna info 0"):
    result = await Runner.run(information_extracter_agent, f"Extract antenna info from {PDF_PATH}", max_turns=30)
    print(result)


Error getting response: Error code: 400 - {'error': {'message': 'Your input exceeds the context window of this model. Please adjust your input and try again.', 'type': 'invalid_request_error', 'param': 'input', 'code': 'context_length_exceeded'}}. (request_id: req_ec2212b7f272fa68a12afea3512750af)


BadRequestError: Error code: 400 - {'error': {'message': 'Your input exceeds the context window of this model. Please adjust your input and try again.', 'type': 'invalid_request_error', 'param': 'input', 'code': 'context_length_exceeded'}}